<a href='https://jupyterhub.user.eopf.eodc.eu/hub/login?next=/hub/spawn?next=/hub/user-redirect/git-pull?repo=https://github.com/eopf-toolkit/eopf-101&branch=main&urlpath=lab/tree/eopf-101/02_about_eopf_zarr/69_coastal_water_dynamics_s1.ipynb#fancy-forms-config={"profile":"choose-your-environment","image":"unlisted_choice","image:unlisted_choice":"4zm3809f.c1.de1.container-registry.ovh.net/eopf-toolkit-python/eopf-toolkit-python:latest","autoStart":"true"}' target="_blank">
  <button style="background-color:#0072ce; color:white; padding:0.6em 1.2em; font-size:1rem; border:none; border-radius:6px; margin-top:1em;">
    🚀 Launch this notebook in JupyterLab
  </button>
</a>

**By:** *[@radosuav](https://github.com/radosuav)*

### Introduction

Monitoring of African rangelands is critical for many reasons, including supporting livestock and pastural communities, mitigating the impact of climate change, conserving wildlife habitats, and combating land degradation and desertification. Effective monitoring with use of Earth Observation requires satellite imagery with both high spatial resolution and high temporal resolution. However, there is currently no single, freely available data source that fulfills these needs. A seamless fusion of data from the Sentinel-2 and Sentinel-3 shortwave optical sensors can meet these monitoring requirements as Sentinel-2 observes at the required spatial resolution (10 m) while Sentinel-3 observes at the required temporal resolution (daily). 

This use-case demonstrates the fusion of Sentinel-2 & Sentinel-3 for more effective rangeland monitoring, especially in regions where the growing season coincides with extended periods of cloud cover. The demonstration shows how Sentinel-2 and Sentinel-3 are accessed efficiently for selected regions of interest using the new zarr format. A time series of Sentinel-2 & Sentinel-3 imagery is then fused using the using a highly efficient and scalable fusion method - [EFAST](https://github.com/DHI-GRAS/efast). EFAST (Efficient Fusion Algorithm across Spatio-Temporal scales) interpolates Sentinel-2 data into smooth time series (both spatially and temporally).<br>
This interpolation is informed by Sentinel-3’s temporal profile such that the phenological changes occurring between two Sentinel-2 acquisitions at a 10 m resolution are assumed to mirror those observed at Sentinel-3’s resolution. The output of the fusion can be used to extract phenological parameters (e.g. growing season min, max, integral). In the [ESA RAMONA](https://www.ramona.earth/) project these parameters were further related to rangeland biomass and health, and used to monitor how these change over time. These data can be used for more effective monitoring programs, planning and policy decisions for African rangelands.

In this notebook we will perform fusion at during the short rainy season (November December) of year 2025 in parts of Serengeti.

### What we will learn

- 🛰️ Accessing Sentinel-2 and Sentinel-3 data via the EODC STAC API using `pystac-client`.
- 🛠️ Preprocess Sentinel-2 and Sentinel-3 imagery including cloud masking, regriding, scaling and georeferencing.
- 🔄 Getting familiar with [EFAST](https://github.com/DHI-GRAS/efast) processing for fusing multi-sensor data to produce high-resolution, frequent time series.

#### Install and import libraries

In [ ]:
!pip install git+https://github.com/DHI-GRAS/efast@python_12
!pip install pyresample session-info rioxarray

In [ ]:
from dateutil import rrule
from datetime import timedelta, datetime
from IPython.display import Image
from pathlib import Path

import numpy as np
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pystac_client import Client
from pyresample import geometry, kd_tree

import efast.efast as efast
import efast.s2_processing as s2
import efast.s3_processing as s3

from zarr_wf_utils import validate_scl
import zarr_efast_utils

import session_info
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pkg_resources")
session_info.show()

For a smooth data retrieval and processing, we set up some required local directories paths

In [ ]:
path = Path("./test_data").absolute()
s3_binning_dir = path / "S3/binning"
s3_composites_dir = path / "S3/composites"
s3_blured_dir = path / "S3/blurred"
s3_calibrated_dir = path / "S3/calibrated"
s3_reprojected_dir = path / "S3/reprojected"
s2_processed_dir = path / "S2/processed"
fusion_dir = path / "fusion_results"

for folder in [
    s3_binning_dir,
    s3_composites_dir,
    s3_blured_dir,
    s3_calibrated_dir,
    s3_reprojected_dir,
    s2_processed_dir,
    fusion_dir,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Sentinel-2 

## Data search

All the data to be processed is openly accesible through the [EOPF STAC Catalog](https://stac.browser.user.eopf.eodc.eu/?.language=en)

In [ ]:
# We first need to establish connection to the EOPF STAC Catalog
eopf_stac_api_root_endpoint = "https://stac.core.eopf.eodc.eu/"
eopf_catalog = Client.open(url=eopf_stac_api_root_endpoint)
eopf_catalog

As we will be searching over Sentinel-2 we need to define the: collection name, time range, area of interest (AOI), tile of interest, bands, and resolution we are interested in. Through `pystac-client` we perform our search over our previously defined `eopf_catalog`

In [ ]:
s2_collection = "sentinel-2-l2a"
date_start = "2024-11-01"
date_end = "2024-12-31"
search_bbox = (33.0, -2.80, 33.99, -1.81)
s2_tile = "T36MWC"
s2_rgb_bands = ["b04", "b03", "b02"]
s2_efast_bands = ["b04", "b8a"]
resolution = 20

In [ ]:
# Search the catalog for items matching the criteria:
s2_l2 = list(
    eopf_catalog.search(
        bbox=search_bbox,
        datetime=f"{date_start}T00:00:00Z/{date_end}T23:59:59Z",
        collections=s2_collection,
    ).item_collection()
)

# Extract the URLs for the product assets from the search results
s2_urls = [
    item.assets["product"].href
    for item in s2_l2
    if f"_{s2_tile}_" in item.assets["product"].href
]

print("Search Results:")
print("Total Items Found for Sentinel-2 over Serengeti:  ", len(s2_urls))

### Data exploration

We can access one item of Sentinel-2 `.zarr` product and extract measurement data for the defined `resolution` and `rgb` bands

In [ ]:
s2_url = s2_urls[-1]

In [ ]:
s2_zarr = xr.open_datatree(s2_url, engine="zarr", chunks={}, decode_timedelta=False)
zarr_meas = s2_zarr.measurements.reflectance[f"r{resolution}m"]

In [ ]:
zarr_meas

In [ ]:
rgb = zarr_meas.to_dataset()[s2_rgb_bands].to_array()
rgb

We can subset the data to a specific AOI using map coordinates (`x: longitude`, `y: latitude`). This will allow us to focus on a small region and visualise efficiently our plots.

In [ ]:
xmin, xmax = 570_000, 580_000
ymin, ymax = 9.70e6, 9.72e6

rgb_subset = rgb.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
rgb_subset.plot.imshow(figsize=(8, 8), robust=True)
plt.title("Sentinel 2 - RGB subset")
plt.axis("off")
plt.show()

## Cloud Masking

Before proceeding with any analysis on Sentinel‑2 imagery, it is important to detect and remove pixels affected by clouds, cloud shadows, because they can distort and contaminate reflectance values, thus leading to unreliable biophysical indicators and spectral analysis. Therefore, we use the [Scene Classification Layer (SCL)](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/scene-classification/) which comes with Sentinel-2 product to detect effected pixels. **SLC** assigns a categorical label to each pixel describing surface type or atmospheric condition (vegetation, water, cloud, cloud shadow, etc.)

In [ ]:
l2a_class = s2_zarr.conditions.mask.l2a_classification[f"r{resolution}m"].scl

Crop the data to a small AOI, and plot it to inspect clouds and surface classes in the scene. 

In [ ]:
slc_subset = l2a_class.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
slc_subset.plot(figsize=(8, 8), robust=True)
plt.title("Sentinel 2 - SLC subset")
plt.axis("off")
plt.show()

Afterwards, we apply the **`validate_scl()`** function, which uses the **SCL** band to create a boolean mask where:
- True: pixel is valid
- False: pixel falls into unwanted classes (no‑data, Saturated defective pixels, shadows, Unclassified, clouds)

**`validate_scl()`** takes the following keyword arguments:

* **`slc`**: `xarray.DataArray` The Scene Classification (SCL) band from a Sentinel‑2 product.

Once the bands are validated we proceed with the masking. Zero is set as `nodata` value is needed for later steps of `efast` `distance_to_clouds` function.


In [ ]:
valid_mask = validate_scl(scl=l2a_class)

band_data = zarr_meas.to_dataset()[s2_rgb_bands]
band_data = xr.where(valid_mask, band_data, 0)

band_data_subset = band_data.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
band_data_subset.to_array().plot.imshow(figsize=(8, 8), robust=True)
plt.title("Sentinel 2 - RGB cloud masked subset")
plt.axis("off")
plt.show()

From the visualized subset, it can be seen that the cloud affected pixels were effectively detected and removed based on the **SCL** information. While the **SCL** mask is not perfect, it provides a robust first order cloud masking that substantially improves the reliability of further spectral analyses.

### EFAST Preprocessing

Now that we learned how to retrieve, explore and apply cloud masking to Sentinel‑2 L2A data, we can prepare the scenes for use with **EFAST** using the function **`s2_preprocess()`** which will perform the following steps for each Sentinel-2 scene:

1. Open the data directly from the cloud-optimized `.zarr` archive.
2. Select the 20 m resolution reflectance group.
3. Extract the **red (B04)** and **near-infrared (B8A)** bands (these bands can be used to calculate Normalized Difference Vegetation Index (NDVI))
4. Load the **SCL** and build a mask to remove clouds and other invalid pixels.
5. Apply this mask to the selected bands, setting invalid pixels to zero.
6. Save the masked bands to a local **GeoTIFF** file in the correct map projection.

<br>
::: {.callout-note}
For details of the code, have a look in the `zarr_efast_utils.py` file.
:::

<br>

The function **`s2_preprocess()`** takes the following arguments:

* **`s2_urls`**: A list of input `.Zarr` URLs for Sentinel‑2 L2A datasets. 
* **`bands`**: A list of Sentinel‑2 spectral band names (e.g., ["b04", "b8a"]).
* **`resolution`**: Spatial resolution of the reflectance grid to load (10, 20, or 60 m). This selects the appropriate sub‑group inside the Zarr hierarchy (e.g., r20m).
* **`output_dir`**: Directory where the processed GeoTIFF files will be written. 

In [ ]:
zarr_efast_utils.s2_preprocess(
    s2_urls=s2_urls, bands=s2_efast_bands, resolution=resolution, output_dir=s2_processed_dir
)

Finally, after all scenes are prepared, we compute the **distance-to-cloud** layer, which is part of the **EFAST** Sentinel-2 processing. This parameter is used because cloud masking can often be inaccurate and the further we are from a detected cloud the more certain we can be of a valid observation.


The function **`distance_to_clouds()`** takes the following arguments:

* **`dir_s2`**: The directory where the Sentinel-2 images are stored. Clouds and shadows should the masked using 0
* **`ratio`**: The (rough) ratio between resolution of Sentinel-2 and Sentinel-3 images. Defaults to 30.
* **`tolerance_percentage`**: Fraction of low-resolution (Sentinel-3) pixel which can be covered by Sentinel-2 resolution cloudy pixels before the low-resolution pixel is considered to be cloudy. Defaults to 0.05.

In [ ]:
s2.distance_to_clouds(dir_s2=s2_processed_dir, ratio=30, tolerance_percentage=0.05)

# Sentinel-3

### Data search

As we did for Sentinel-2 collection, we will now retrieve the Sentinel-3 OLCI items.
The collection name, bands and AOI are once more defined to search over the `sentinel-3-olci-l2-lfr` collection inside the **EOPF STAC Catalog**.

In [ ]:
s3_collection = "sentinel-3-olci-l2-lfr"
s3_bands = ["rc681", "rc865"]

In [ ]:
# Search the catalog for items matching the criteria:
s3_l2 = list(
    eopf_catalog.search(
        bbox=search_bbox,
        datetime=f"{date_start}T00:00:00Z/{date_end}T23:59:59Z",
        collections=s3_collection,
        query={"product:timeliness_category": {"eq": "NT"}},
    ).item_collection()
)

# Extract the URLs for the product assets from the search results
s3_urls = [item.assets["product"].href for item in s3_l2]

print("Search Results:")
print("Total Items Found for Sentinel-3 OLCI over Serengeti:  ", len(s3_urls))

<br>
::: {.callout-note}
As the EOPF STAC Catalog, provides a sample data, some scenes are not directly available thorugh the Catalogs search. The required Sentinel-3 items are provided **here** directly as URLs.
:::
<br>

In [ ]:
# Temporary, until STAC catalogue is fixed
s3_urls = ['https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241101T071724_20241101T072024_20250805T204152_0179_118_334_3060_ESA_R_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241104T073951_20241104T074251_20250805T234912_0179_118_377_3060_ESA_R_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241218T065836_20241218T070136_20241219T073531_0179_120_234_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241201T070106_20241201T070406_20241201T203734_0179_100_234_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241219T073443_20241219T073743_20241219T221414_0179_101_106_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241111T071646_20241111T071946_20241111T222052_0179_099_334_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241201T073948_20241201T074248_20241202T081044_0179_119_377_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241201T073648_20241201T073948_20241202T081046_0179_119_377_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241213T072532_20241213T072832_20241214T081519_0179_120_163_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241220T074714_20241220T075014_20241221T081623_0179_120_263_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241203T074942_20241203T075242_20241203T223738_0179_100_263_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241216T074758_20241216T075058_20241217T081943_0179_120_206_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241101T071424_20241101T071724_20250805T204149_0179_118_334_2880_ESA_R_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241104T073651_20241104T073951_20250805T234733_0179_118_377_2880_ESA_R_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241127T070451_20241127T070751_20241127T220949_0179_100_177_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241123T074717_20241123T075017_20241124T083043_0179_119_263_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241219T073143_20241219T073443_20241219T221408_0179_101_106_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241213T072832_20241213T073132_20241214T081534_0179_120_163_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241202T071337_20241202T071637_20241203T074421_0180_120_006_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241210T070606_20241210T070906_20241211T073745_0179_120_120_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241203T074642_20241203T074942_20241203T223732_0179_100_263_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241111T075530_20241111T075830_20241112T083049_0179_119_092_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241204T072031_20241204T072331_20241204T220357_0179_100_277_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241229T071337_20241229T071637_20241230T074406_0179_121_006_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241214T070221_20241214T070521_20241215T073651_0179_120_177_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241205T073303_20241205T073603_20241206T081002_0179_120_049_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241212T071302_20241212T071602_20241212T220350_0180_101_006_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241224T070449_20241224T070749_20241224T220506_0179_101_177_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241103T072416_20241103T072716_20241103T221756_0179_099_220_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241128T071421_20241128T071721_20241129T074602_0179_119_334_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241228T070104_20241228T070404_20241228T211722_0179_101_234_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241216T075058_20241216T075358_20241217T081946_0179_120_206_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241111T075830_20241111T080130_20241112T083049_0179_119_092_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241204T072331_20241204T072631_20241204T220403_0179_100_277_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241211T074213_20241211T074513_20241211T220921_0179_100_377_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241205T073603_20241205T073903_20241206T081002_0179_120_049_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241206T070952_20241206T071252_20241207T074238_0179_120_063_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241211T073913_20241211T074213_20241211T220916_0179_100_377_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241128T071721_20241128T072021_20241129T074602_0180_119_334_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241207T074258_20241207T074558_20241207T215909_0179_100_320_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241122T073446_20241122T073746_20241122T220827_0179_100_106_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241127T074332_20241127T074632_20241128T081434_0180_119_320_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241114T074212_20241114T074512_20241114T221534_0179_099_377_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241207T074558_20241207T074858_20241207T215915_0179_100_320_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241112T072919_20241112T073219_20241113T081748_0179_119_106_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241112T073219_20241112T073519_20241113T081749_0179_119_106_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241208T071647_20241208T071947_20241208T215638_0179_100_334_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241208T071947_20241208T072247_20241208T215643_0179_100_334_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241127T074032_20241127T074332_20241128T081433_0179_119_320_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241124T071806_20241124T072106_20241125T075255_0179_119_277_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241113T070609_20241113T070909_20241114T074818_0179_119_120_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241114T073912_20241114T074212_20241114T221528_0179_099_377_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241126T073102_20241126T073402_20241126T222646_0179_100_163_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241125T075413_20241125T075713_20241125T203834_0179_100_149_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241115T075446_20241115T075746_20241116T082753_0179_119_149_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241115T075146_20241115T075446_20241116T082746_0179_119_149_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241224T074031_20241224T074331_20241225T081404_0179_120_320_2880_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241108T073605_20241108T073905_20241109T080650_0179_119_049_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241126T072802_20241126T073102_20241126T222640_0179_100_163_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241125T075713_20241125T080013_20241125T203839_0179_100_149_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241115T071602_20241115T071902_20241115T203917_0179_100_006_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241115T071302_20241115T071602_20241115T203911_0179_100_006_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241228T073648_20241228T073948_20241229T081109_0180_120_377_2880_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241123T070836_20241123T071136_20241123T221120_0179_100_120_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241208T075829_20241208T080129_20241209T082958_0180_120_092_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241208T075529_20241208T075829_20241209T082959_0179_120_092_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241227T072715_20241227T073015_20241227T212122_0179_101_220_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241116T072536_20241116T072836_20241117T080953_0179_119_163_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241231T072330_20241231T072630_20241231T221210_0179_101_277_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241117T070225_20241117T070525_20241118T073156_0179_119_177_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241119T071219_20241119T071519_20241119T211728_0179_100_063_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241224T074331_20241224T074631_20241225T081405_0179_120_320_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241227T072415_20241227T072715_20241227T212116_0179_101_220_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241119T074802_20241119T075102_20241120T083445_0179_119_206_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241104T073951_20241104T074251_20241105T081341_0179_118_377_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241104T073651_20241104T073951_20241105T081341_0179_118_377_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241212T075143_20241212T075443_20241213T082646_0180_120_149_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241108T073305_20241108T073605_20241109T080655_0179_119_049_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241116T072836_20241116T073136_20241117T081013_0179_119_163_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241231T072030_20241231T072330_20241231T221205_0179_101_277_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241118T073530_20241118T073830_20241118T224144_0179_100_049_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241118T073830_20241118T074130_20241118T224149_0179_100_049_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241129T075028_20241129T075328_20241129T211722_0179_100_206_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241122T073146_20241122T073446_20241122T220822_0179_100_106_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241104T070105_20241104T070405_20241104T221519_0179_099_234_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241212T075443_20241212T075743_20241213T082657_0179_120_149_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241105T071340_20241105T071640_20241106T075704_0179_119_006_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241106T074943_20241106T075243_20241106T211630_0179_099_263_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241217T072447_20241217T072747_20241218T075502_0179_120_220_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241102T075026_20241102T075326_20241102T211206_0180_099_206_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241121T065840_20241121T070140_20241122T072648_0179_119_234_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241106T074643_20241106T074943_20241106T211624_0179_099_263_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241119T075102_20241119T075402_20241120T083444_0179_119_206_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241123T074417_20241123T074717_20241124T083019_0179_119_263_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241217T072147_20241217T072447_20241218T075511_0179_120_220_2880_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241101T071724_20241101T072024_20241102T074727_0179_118_334_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241101T071424_20241101T071724_20241102T074730_0179_118_334_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241102T075326_20241102T075626_20241102T211211_0179_099_206_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241103T072716_20241103T073016_20241103T221802_0179_099_220_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241107T072032_20241107T072332_20241107T204607_0179_099_277_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241107T072332_20241107T072632_20241107T204613_0179_099_277_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241209T073218_20241209T073518_20241210T080704_0179_120_106_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241109T070954_20241109T071254_20241110T074326_0179_119_063_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241110T074257_20241110T074557_20241110T220753_0180_099_320_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241110T074557_20241110T074857_20241110T220758_0179_099_320_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241120T072151_20241120T072451_20241121T075845_0179_119_220_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241120T072451_20241120T072751_20241121T075848_0179_119_220_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241209T072918_20241209T073218_20241210T080702_0179_120_106_2880_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241111T071946_20241111T072246_20241111T222058_0180_099_334_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241129T075328_20241129T075628_20241129T211733_0180_100_206_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241130T072417_20241130T072717_20241130T220448_0179_100_220_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241215T073528_20241215T073828_20241215T213059_0179_101_049_2880_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241216T071217_20241216T071517_20241216T220346_0179_101_063_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241225T071721_20241225T072021_20241226T074943_0179_120_334_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241230T074641_20241230T074941_20241230T215402_0179_101_263_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241221T072104_20241221T072404_20241222T075414_0180_120_277_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241124T072106_20241124T072406_20241125T075257_0179_119_277_3060_PS1_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241130T072717_20241130T073017_20241130T220453_0179_100_220_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241212T071602_20241212T071902_20241212T220355_0179_101_006_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241215T073828_20241215T074128_20241215T213105_0179_101_049_3060_PS2_O_NT_002.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241230T074941_20241230T075241_20241230T215407_0179_101_263_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241225T071421_20241225T071721_20241226T074940_0179_120_334_2880_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241220T070832_20241220T071132_20241220T222228_0179_101_120_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241220T074414_20241220T074714_20241221T081629_0179_120_263_2880_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241221T071804_20241221T072104_20241222T075421_0179_120_277_2880_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241222T075710_20241222T080010_20241222T211518_0180_101_149_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241222T075410_20241222T075710_20241222T211513_0179_101_149_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241226T075026_20241226T075326_20241226T212416_0179_101_206_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241223T072759_20241223T073059_20241223T220348_0179_101_163_2880_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3A_OL_2_LFR____20241228T073948_20241228T074248_20241229T081110_0179_120_377_3060_PS1_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241226T075326_20241226T075626_20241226T212422_0179_101_206_3060_PS2_O_NT_003.zarr', 'https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/tutorial_data/cpm_v262/S3B_OL_2_LFR____20241223T073059_20241223T073359_20241223T220354_0179_101_163_3060_PS2_O_NT_003.zarr']
print("Total Items Found for Sentinel-3 OLCI over Serengeti:  ", len(s3_urls))

### Data exploration

We access a Sentinel-3 OLCI scene from the `.zarr` archive and inspect its structure and available variables.

In [ ]:
s3_url = s3_urls[0]
s3_zarr = xr.open_datatree(s3_url, engine="zarr")

s3_zarr[s3_zarr.groups[0]].groups

In [ ]:
s3_zarr[s3_zarr.groups[2]].data_vars

As **red** and **near-infrared** reflectance bands are used by **EFAST**, we retrieve them through the `measurements` group.

In [ ]:
band_data_s3 = s3_zarr.measurements.to_dataset()[s3_bands]
band_data_s3

And to visualize a subset of the retrieved items:

In [ ]:
band_data_s3.rc681[::4, ::4].plot(figsize=(8, 6))

### Regredding

#### Define the target regular grid

Sentinel-3 data is provided in a **swath geometry** and not on a regular grid. Each pixel has its own latitude and longitude.
To be able to use the data, we need to resample it onto a regular lat/lon grid over our area of interest. For this, we can make use of the utility [geometry.AreaDefinition](https://pyresample.readthedocs.io/en/latest/howtos/geo_def.html) as part of `pyresample` library.

The function **`geometry.AreaDefinition()`** takes the following arguments:

* **`area_id`**: ID of area
* **`description`**: Description for the grid.
* **`proj_id`**: ID of projection.
* **`projection`**: A PROJ dictionary, PROJ/WKT string, or a CRS object defining the coordinate reference system.
* **`width`**: Number of pixels in the x‑direction (grid columns).
* **`height`**: Number of pixels in the y‑direction (grid rows).
* **`area_extent`**: Spatial bounding box in projection coordinates, given as (lower_left_x, lower_left_y, upper_right_x, upper_right_y).

In [ ]:
grid = geometry.AreaDefinition(
    area_id="olci_grid",
    description="OLCI projected grid",
    proj_id="latlon",
    projection="EPSG:4326",
    width=int((search_bbox[2] - search_bbox[0]) / 0.0027),  
    height=int((search_bbox[3] - search_bbox[1]) / 0.0027),  
    area_extent=search_bbox,
)

grid

#### Resample from **swath** to **grid**

We resample the irregular swath data to the regular grid using nearest-neighbor interpolation. 

First, we define the swath geometry:

In [ ]:
s3_swath = geometry.SwathDefinition(
    lons=band_data_s3["longitude"].values * 1_000_000,
    lats=band_data_s3["latitude"].values * 1_000_000,
)

Then we resample all selected OLCI bands onto the grid using the `resample_nearest()` function from `pyresample` library.

The function **`resample_nearest()`** performs nearest‑neighbor interpolation using a `KD‑tree` search and accepts the following arguments:

* **`source_geo_def`**: Geometry of the source swath (e.g., longitudes and latitudes of OLCI pixels).
* **`data`**: Either a 1D array of pixel values or a 2D array with multiple channels stacked along the last dimension.
* **`target_geo_def`**: The target grid onto which data will be resampled (e.g., the AreaDefinition grid).
* **`radius_of_influence`**: Maximum search distance in meters. Only source pixels within this radius are considered.
* **`fill_value`**:  Value assigned to pixels with no valid neighbor within the radius. If None, a masked array is returned.

In [ ]:
resampled = kd_tree.resample_nearest(
    source_geo_def=s3_swath,
    data=np.stack([band_data_s3[band].values for band in s3_bands], axis=2),
    target_geo_def=grid,
    radius_of_influence=500,
    fill_value=np.nan,
)

### Georefrecing 

Once we obtain a regular grid, we build a properly georeferenced-`xarray` object with **x**/**y** coordinates and our target **CRS** (EPSG:4326).

In [ ]:
resampled = np.transpose(resampled, (2, 0, 1))

# Build coordinate arrays
x_res = (grid.area_extent[2] - grid.area_extent[0]) / grid.width
y_res = (grid.area_extent[3] - grid.area_extent[1]) / grid.height
x_coords = np.linspace(grid.area_extent[0] + x_res / 2, grid.area_extent[2] - x_res / 2, grid.width)
y_coords = np.linspace(grid.area_extent[3] - y_res / 2, grid.area_extent[1] + y_res / 2, grid.height)

band_data_s3_proj = xr.DataArray(
    resampled,
    dims=("band", "y", "x"),
    coords={"band": s3_bands, "y": y_coords, "x": x_coords},
)

band_data_s3_proj.rio.write_crs("EPSG:4326", inplace=True)
band_data_s3_proj

To have a quick overview of our results:

In [ ]:
band_data_s3_proj.sel(band=s3_bands[0]).plot(figsize=(8, 6))
plt.title("Sentinel-3 red band (georeferenced and resampled)")
plt.show()

## Bundle Preprocessing

So far we have walked through each processing of Sentinel-3 OLCI step separately. To automate the workflow and apply it efficiently across Sentinel-3 acquisitions, we could now wrap all these operations into a single processing function called `s3_preprocess` to prepare the Sentinel-3 scenes for use with **EFAST**. 

For each Sentinel-3 scene, `s3_preprocess` performs the following steps:

1. Open the data directly from the cloud-optimized Zarr archive.
2. Select the **red (681 nm)** and **near-infrared (865 nm)** reflectance bands.
3. Use the per-pixel latitude and longitude to define the original swath geometry.
4. Define a **regular latitude/longitude grid** over the area of interest.
5. Resample the swath data onto this grid using **nearest-neighbor** interpolation.
6. Apply the provided **scale factors** to obtain physical reflectance values.
7. Build a georeferenced data cube with proper coordinates and CRS.
8. Save the result as a compressed **GeoTIFF** file for later use by EFAST.

For details of the code, see `zarr_efast_utils.py` file. The function **`s3_preprocess()`** takes the following arguments:

* **`s3_urls`**: A list of input `.Zarr` URLs for Sentinel‑3 datasets. 
* **`bands`**: A list of Sentinel‑3 spectral band names (e.g., ["rc681", "rc865"]).
* **`search_bbox`**: Spatial bounding box in projection coordinates, given as (lower_left_x, lower_left_y, upper_right_x, upper_right_y).
* **`target_resolution_deg`**: Target spatial resolution in degrees. 
* **`output_dir`**: Directory where the processed GeoTIFF files will be written. 

In [ ]:
s3_bands = ["rc681", "rc865"]
print(s3_bands, search_bbox, s3_binning_dir)

zarr_efast_utils.s3_preprocess(
    s3_urls=s3_urls,
    bands=s3_bands,
    search_bbox=search_bbox,
    target_resolution_deg=0.0027,
    output_dir=s3_binning_dir,
)

Once the Sentinel‑3 scenes have been resampled to the target grid, **weighted temporal composites** are generated using the `produce_median_composite()` function. Each composite represents a time‑window summary of several OLCI observations, taking into account temporal distance from the central date as well as proximity to clouds.

The function `produce_median_composite()` accepts the following arguments:

*   **`dir_s3`**: Directory containing the resampled Sentinel‑3 scenes.
*   **`composite_dir`**: Directory where the output composite images are written.
*   **`step`**: Temporal spacing (in days) between consecutive composites.
*   **`mosaic_days`**: Width of the temporal window used to collect images for each composite.
*   **`s3_bands`**: List of OLCI bands to include. If `None`, all available bands are used.
*   **`D`**: Distance‑to‑cloud cutoff in pixels; beyond this distance, clouds no longer affect pixel weighting.
*   **`sigma_doy`**: Standard deviation controlling the temporal weighting around the central date.

In [ ]:
# Define settings
mosaic_days = 60 
step = 10 

In [ ]:
s3.produce_median_composite(
    dir_s3=s3_binning_dir,
    composite_dir=s3_composites_dir,
    mosaic_days=mosaic_days,
    step=step,
    s3_bands=None,
)

After creating the composite time series, an optional **Gaussian smoothing** is applied to reduce noise while preserving the large scale spatial structure. 

This is done with `smoothing()`, which takes:

*   **`s3_dir`**: Directory containing the composite images.
*   **`smoothed_dir`**: Directory where the smoothed outputs will be stored.
*   **`product`**: String used to select which files to smooth (default: `"composite"`).
*   **`std`**: Standard deviation of the 2D Gaussian kernel.
*   **`preserve_nan`**: Whether invalid pixels (`NaN`) should remain unchanged during smoothing.

In [ ]:
s3.smoothing(
    s3_composites_dir,
    s3_blured_dir,
    std=1,
    preserve_nan=False,
)

The function `reformat_s3()` is then used to reformat Sentinel-3 images to a format suitable for analysis. and Reformat Sentinel-3 images to a format suitable for analysis. In addition `reproject_and_crop_s3()` function will create single-band composites of Sentinel-3 data.

In [ ]:
s3.reformat_s3(
    s3_blured_dir,
    s3_calibrated_dir,
)
s3.reproject_and_crop_s3(
    s3_calibrated_dir,
    s2_processed_dir,
    s3_reprojected_dir,
)

## EFAST Sentinel -2 and Sentinel -3 Fusion

Finally, to generate a synthetic high‑resolution Sentinel‑2–like image on each prediction date, the **EFAST fusion** algorithm is then applied. The `fusion()` function combines information from the low‑resolution from Sentinel‑3 composites and the high‑resolution from Sentinel‑2 observations that fall within a defined temporal window. The method uses temporal weighting, similarity scores, and cloud‑distance weighting to produce a fused high‑resolution reflectance product.

The function `fusion()` takes several argumnets with the ones that will be used are:

*   **`pred_date`**: The target date for which the fused high‑resolution image will be produced.
*   **`lr_dir`**: Directory containing the low‑resolution (Sentinel‑3) inputs.
*   **`hr_dir`**: Directory containing the high‑resolution (Sentinel‑2) preprocessed images.
*   **`fusion_dir`**: Output directory for the fused products.
*   **`product`**: Keyword used to select the Sentinel‑2 files participating in the fusion.
*   **`max_days`**: Maximum temporal distance allowed between the acquisition date and the prediction date.
*   **`ratio`**: Conversion factor between high and low‑resolution pixel sizes (S2 vs. S3).
*   **`minimum_acquisition_importance`**: Minimum weight threshold for including a high‑resolution acquisition in the fusion.

In [ ]:
# Define EFAST settings
maximum_fusion_days = 30
ratio = 30 
# Note: `mosaic_days` and `step` settings are used from the last mosaic operation

In [ ]:
# Perform EFAST fusion
for date in rrule.rrule(
    rrule.DAILY,
    dtstart=datetime.strptime(date_start, "%Y-%m-%d") + timedelta(step),
    until=datetime.strptime(date_end, "%Y-%m-%d") - timedelta(step),
    interval=step,
):
    efast.fusion(
        pred_date=date,
        lr_dir=s3_reprojected_dir,
        hr_dir=s2_processed_dir,
        fusion_dir=fusion_dir,
        product="REFL",
        ratio=ratio,
        max_days=maximum_fusion_days,
        minimum_acquisition_importance=0,
    )

## Fusion results

Once the **EFAST** fusion is complete, we can inspect the results and visualize them as a spatio-temporal time series.

Firstly, we need to collect all fusion result files, extract the acquisition date from the filename, and sort them in chronological order.

In [ ]:
# Get all the TIFF files
tif_files = list(fusion_dir.glob("*.tif"))

# Get dates and sort files by date
file_dates = []

for tif_file in tif_files:
    date_str = tif_file.stem.split("_")[1]
    date = datetime.strptime(date_str, "%Y%m%d")
    file_dates.append((date, tif_file))
file_dates.sort(key=lambda x: x[0])

Afterwards, we build a time series data cube over the predefined subset coordinates (`xmin`, `xmax`; `ymin`, `ymax`).

In [ ]:
dates = [d for d, f in file_dates]
files = [f for d, f in file_dates]

# Open the files and stack the data along time dimension
da_list = []

for date, tif in zip(dates, files):  
    da = xr.open_dataarray(tif)
    da_subset = da.sel(band=2).sel(
        x=slice(xmin, xmax), y=slice(ymax, ymin)
    )
    da_subset = da_subset.expand_dims(time=[date])
    da_list.append(da_subset)

stack = xr.concat(da_list, dim="time")
stack

The generated maps show reflectance in the **NIR** band. Higher values of it indicate presence of dense green vegetation.

In [ ]:
stack.plot.imshow(col="time", col_wrap=3, robust=True)

To visualize the dynamic changes across the time series, we can generate an animated GIF from the fused product.

In [ ]:
# Global contrast stretch ---
vmin, vmax = np.nanpercentile(stack.values, (2, 98))

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(
    stack.isel(time=0).values,
    vmin=vmin,
    vmax=vmax,
)
title = ax.set_title(str(stack.time.values[0])[:10])
ax.axis("off")
plt.colorbar(im, ax=ax, label="Reflectance")

def func(frame):
    im.set_data(stack.isel(time=frame).values)
    title.set_text(str(stack.time.values[frame])[:10])
    return im, title

ani = animation.FuncAnimation(
    fig,
    func,
    frames=stack.sizes["time"],
    interval=400,
    blit=True,
)
plt.close(fig)
ani.save(
    "efast_ts.gif",
    writer="pillow",
    fps=2
)

In [ ]:
Image(filename="efast_ts.gif")

## 💪 Now it is your turn 

Congratulations 🎉 We have worked through the complete workflow for fusing Sentinel-2 and Sentinel-3 to produce frequent, high-resolution images from `.zarr` data. Now it is your turn to explore and expand the analysis in the following ways:

### Task 1: Explore your own area of interest

Choose a different rangeland anywhere in the world. Use different STAC search configurations (`date_start`, `date_end`, `search_bbox`)

###  Task 2: Experiment with different EFAST Sentinel-3 preprocessing settings.

* `mosaic_days`
* `step`
* `ratio` 

### Task 3: Analyze rangeland dynamics using the EFAST fused time series

* Use the EFAST fused product to analyze vegetation dynamics over rangelands.
* Compute vegetation indices (e.g. NDVI) from the fused time series.
* Explore seasonal patterns, trends, or anomalies in rangeland condition using the fused high-resolution time series.

## Conclusion

In this notebook, we demonstrated how to use Sentinel-2 and Sentinel-3 data in `.zarr` format together with EFAST to generate frequent, high-resolution time series over rangeland areas. The Zarr data format enables efficient access to large satellite archives without loading entire datasets into memory.

We developed a complete processing workflow using pystac-client and the EOPF STAC API to discover and access the data, followed by sensor-specific preprocessing steps. For Sentinel-2, this included band selection and cloud masking. For Sentinel-3, this included resampling the swath data onto a regular grid. The preprocessed datasets were then fused using [EFAST](https://github.com/DHI-GRAS/efast) to produce a consistent, analysis-ready time series with both high spatial and temporal resolution.

Finally, we showed how to inspect and visualize the fused results as a spatio-temporal data cube and how to explore temporal dynamics over rangelands using time series plots and animations.

<hr>

## What's next?

This resource is constantly updated! Stay Tuned for new chapters 🛰️ !